In [3]:
import numpy as np
import pandas as pd

from pathlib import Path
import joblib
import sys

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)
from imblearn.over_sampling import SMOTE

from src.preprocessing import load_and_clean_data
from src.feature_engineering import build_preprocessor

In [4]:
# Project paths
ROOT_DIR = Path.cwd().parent

DATA_PATH =ROOT_DIR / "notebooks" / "churn_preprocessed.csv"
MODEL_DIR = ROOT_DIR / "models"
MODEL_PATH = MODEL_DIR / "churn_model.joblib"

In [5]:
df = pd.read_csv(DATA_PATH)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [6]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)


In [8]:
# Build preprocessing + model pipeline
model_LogReg = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("smote", SMOTE(random_state=42)),
    ("LogReg", LogisticRegression(max_iter=1000, random_state=42))
])

In [9]:
# Build preprocessing + model pipeline
model_LogReg_balanced = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("smote", SMOTE(random_state=42)),
    ("LogReg balanced", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])

In [10]:
# Build preprocessing + model pipeline
model_Random_Forest = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("smote", SMOTE(random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1))
    ])

In [11]:
# Build preprocessing + model pipeline
model_XGBoost = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("smote", SMOTE(random_state=42)),
    ("XGBoost", XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=-1))
    ])

In [12]:
# Create the same folds for every model
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
# Define evaluation metrics
scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [ ]:
# Run cross-validation
cv_LogReg = cross_validate(
    model_LogReg,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [33]:
# Run cross-validation
cv_LogReg_balanced = cross_validate(
    model_LogReg_balanced,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [34]:
# Run cross-validation
cv_Random_Forest = cross_validate(
    model_Random_Forest,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [35]:
# Run cross-validation
cv_XGBoost = cross_validate(
    model_XGBoost,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [36]:
# Display mean scores across folds
results = pd.DataFrame({
    "Metric": ["ROC-AUC", "precision", "Recall", "F1"],
    "Logistic Regression Mean Score": [cv_LogReg["test_roc_auc"].mean(), cv_LogReg["test_precision"].mean(), cv_LogReg["test_recall"].mean(), cv_LogReg["test_f1"].mean()],
    "Logistic Regression std": [cv_LogReg["test_roc_auc"].std(), cv_LogReg["test_precision"].std(), cv_LogReg["test_recall"].std(), cv_LogReg["test_f1"].std()],
    
    "Logistic Regression Balanced Mean Score": [cv_LogReg_balanced["test_roc_auc"].mean(), cv_LogReg_balanced["test_precision"].mean(), cv_LogReg_balanced["test_recall"].mean(), cv_LogReg_balanced["test_f1"].mean()],
    "Logistic Regression Balanced std": [cv_LogReg_balanced["test_roc_auc"].std(), cv_LogReg_balanced["test_precision"].std(), cv_LogReg_balanced["test_recall"].std(), cv_LogReg_balanced["test_f1"].std()],

    "Random Forest Mean Score": [cv_Random_Forest["test_roc_auc"].mean(), cv_Random_Forest["test_precision"].mean(), cv_Random_Forest["test_recall"].mean(), cv_Random_Forest["test_f1"].mean()],
    "Random Forest std": [cv_Random_Forest["test_roc_auc"].std(), cv_Random_Forest["test_precision"].std(), cv_Random_Forest["test_recall"].std(), cv_Random_Forest["test_f1"].std()],
    
    "XGBoost Mean Score": [cv_XGBoost["test_roc_auc"].mean(), cv_XGBoost["test_precision"].mean(), cv_XGBoost["test_recall"].mean(), cv_XGBoost["test_f1"].mean()],
    "XGBoost std": [cv_XGBoost["test_roc_auc"].std(), cv_XGBoost["test_precision"].std(), cv_XGBoost["test_recall"].std(), cv_XGBoost["test_f1"].std()]
})

display(results.round(4))

,Metric,Logistic Regression Mean Score,Logistic Regression std,Logistic Regression Balanced Mean Score,Logistic Regression Balanced std,Random Forest Mean Score,Random Forest std,XGBoost Mean Score,XGBoost std
0,ROC-AUC,0.8450,0.0131,0.8450,0.0131,0.8314,0.0086,0.8419,0.0100
1,precision,0.5272,0.0162,0.5272,0.0162,0.5966,0.0227,0.6122,0.0202
2,Recall,0.7860,0.0376,0.7860,0.0376,0.6140,0.0136,0.6174,0.0271
3,F1,0.6308,0.0209,0.6308,0.0209,0.6051,0.0164,0.6144,0.0174


In [37]:
# Train
model_LogReg.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


In [38]:
# 6. Predict
y_prob = model_LogReg.predict_proba(X_test)[:, 1]

# Choose a threshold
threshold = 0.7

# Convert probabilities to predictions
y_pred_threshold = (y_prob >= threshold).astype(int)

In [39]:
# 7. Evaluate
print("Accuracy:", round(accuracy_score(y_test, y_pred_threshold), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_threshold))

Accuracy: 0.7906
ROC-AUC: 0.8402

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.86      0.86      1035
           1       0.61      0.59      0.60       374

    accuracy                           0.79      1409
   macro avg       0.73      0.73      0.73      1409
weighted avg       0.79      0.79      0.79      1409



In [ ]:
# 8. Save the complete pipeline
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model_LogReg, MODEL_PATH)

print(f"\nModel saved to: {MODEL_PATH}")



Model saved to: d:\Worrrk\Python_Projects\Customer_Churn_Project\models\churn_model.joblib
